In [9]:
%pip install langchain chromadb huggingface tiktoken pypdf langchain_huggingface langchain-community sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 9.0 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [5]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [6]:
docs=[doc1, doc2, doc3, doc4, doc5]

In [11]:
vector_store=Chroma(
    embedding_function=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2'),
    persist_directory='chroma_db',
    collection_name='sample'
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8344.86it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/var/folders/k0/cph1_kvx2bb0d_gdrrg0l94m0000gn/T/ipykernel_25623/156744037.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store=Chroma(


In [12]:
vector_store.add_documents(docs)

['15ec4105-5f47-4742-aece-78b1f31e797e',
 'c6d2cdc7-7b6c-49e2-87ce-914aac7d1b9b',
 '90aae799-d2db-41f1-b992-1ce479491f21',
 '58a8c68d-614f-49ec-abd3-cd468b7c6016',
 'fa972a08-18ae-4183-9b98-f586e05e6b58']

In [13]:
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['15ec4105-5f47-4742-aece-78b1f31e797e',
  'c6d2cdc7-7b6c-49e2-87ce-914aac7d1b9b',
  '90aae799-d2db-41f1-b992-1ce479491f21',
  '58a8c68d-614f-49ec-abd3-cd468b7c6016',
  'fa972a08-18ae-4183-9b98-f586e05e6b58'],
 'embeddings': array([[ 0.0099472 ,  0.06914334, -0.05147113, ..., -0.03543335,
          0.01284803,  0.01248282],
        [ 0.00127749,  0.03129849, -0.02375378, ..., -0.00518365,
         -0.03280605,  0.02737715],
        [-0.10265918,  0.02650805,  0.02271504, ..., -0.03359751,
         -0.07984944, -0.0150771 ],
        [ 0.02123396, -0.02468552, -0.04494364, ..., -0.10995807,
          0.00572558,  0.09915375],
        [ 0.01873974,  0.04382839, -0.04304254, ..., -0.07801615,
         -0.07840678, -0.0030419 ]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [18]:
vector_store.similarity_search(
    query='Who among these are bowler',
    k=1
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [19]:
vector_store.similarity_search_with_score(
    query='Who among are bowler?',
    k=2
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9359544515609741),
 (Document(metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.1506586074829102)]

In [22]:
vector_store.similarity_search_with_score(
    query='',
    filter={'team':'Chennai Super Kings'}
) 

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436007499694824),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.890937328338623)]

In [25]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='15ec4105-5f47-4742-aece-78b1f31e797e', document=updated_doc1)


In [26]:
vector_store.get()

{'ids': ['15ec4105-5f47-4742-aece-78b1f31e797e',
  'c6d2cdc7-7b6c-49e2-87ce-914aac7d1b9b',
  '90aae799-d2db-41f1-b992-1ce479491f21',
  '58a8c68d-614f-49ec-abd3-cd468b7c6016',
  'fa972a08-18ae-4183-9b98-f586e05e6b58'],
 'embeddings': None,
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
  "Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to 

In [29]:
vector_store.delete(ids=['15ec4105-5f47-4742-aece-78b1f31e797e'])

In [30]:
vector_store.get()

{'ids': ['c6d2cdc7-7b6c-49e2-87ce-914aac7d1b9b',
  '90aae799-d2db-41f1-b992-1ce479491f21',
  '58a8c68d-614f-49ec-abd3-cd468b7c6016',
  'fa972a08-18ae-4183-9b98-f586e05e6b58'],
 'embeddings': None,
 'documents': ["Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.',
  'Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas':